In [ ]:
# =====================================================================
# =====================================================================
#
#       1D NEUTRON DIFFRACTION MATERIAL CLASSIFIER
#
#       PyTorch Residual CNN
#
#       INPUT:
#           1D diffraction pattern I(2theta)
#
#       OUTPUT:
#           Material class
#
#       Includes:
#
#           ✓ automatic dataset discovery
#           ✓ data validation
#           ✓ leakage-resistant grouped splitting
#           ✓ train / validation / test sets
#           ✓ robust intensity preprocessing
#           ✓ training-only normalization
#           ✓ mild physics-compatible augmentation
#           ✓ residual 1D CNN
#           ✓ squeeze-and-excitation channel attention
#           ✓ dropout
#           ✓ AdamW
#           ✓ label smoothing
#           ✓ class weighting if necessary
#           ✓ gradient clipping
#           ✓ LR scheduler
#           ✓ early stopping
#           ✓ best-model checkpoint
#           ✓ CUDA AMP when available
#           ✓ Apple Silicon MPS support
#           ✓ accuracy / macro F1 / balanced accuracy
#           ✓ Top-2 accuracy
#           ✓ confusion matrices
#           ✓ classification report
#           ✓ calibration error
#           ✓ training curves
#           ✓ saved test predictions
#
# =====================================================================
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

from pathlib import Path
import json
import math
import random
import time
import contextlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        f1_score,
        classification_report,
        confusion_matrix
    )
except ImportError:
    raise ImportError(
        "\nscikit-learn is required for the evaluation metrics.\n"
        "Install it in the active environment with:\n\n"
        "    conda install scikit-learn\n"
        "\nor:\n\n"
        "    pip install scikit-learn\n"
    )


# =====================================================================
# 1. USER SETTINGS
# =====================================================================


# ---------------------------------------------------------------------
# DATASET
# ---------------------------------------------------------------------
#
# This is the root created by the McStas/NCrystal simulation notebook.
#
#
# ---------------------------------------------------------------------

# =====================================================================
# PROJECT ROOT
# =====================================================================

def find_project_root():

    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents
    ]:

        if (
            (candidate / "README.md").is_file()
            and
            (candidate / "notebooks").is_dir()
        ):

            return candidate


    raise RuntimeError(
        "\nCould not locate the project root."
    )


PROJECT_ROOT = find_project_root()


DATASET_ROOT = (
    PROJECT_ROOT
    /
    "data"
    /
    "multi_material_diffraction_dataset"
)


print("Project root:")
print(PROJECT_ROOT)

print()

print("Dataset root:")
print(DATASET_ROOT)

print()

# ---------------------------------------------------------------------
# ---------------------------------------------------------------------

DATASET_SWEEP = None


# ---------------------------------------------------------------------
# WHICH 1D SIGNAL TO USE?
# ---------------------------------------------------------------------
#
# Available from our simulation pipeline:
#
#       "intensity_mean"
#       "intensity_sum"
#       "intensity_mean_unitmax"
#
#
# Recommendation:
#
#       intensity_mean
#
# We then perform our own robust normalization below.
#
# ---------------------------------------------------------------------

PROFILE_KEY = "intensity_mean"


# =====================================================================
# 2. SPLITTING
# =====================================================================
#
# IMPORTANT:
#
# We split by complete (lambda, dlambda) CONDITIONS.
#
# Example:
#
#       lambda = 2.1
#       dlambda = 0.03
#
# if assigned to TEST:
#
# that condition is NOT present in training for ANY material.
#
# This reduces leakage between neighboring synthetic simulations.
#
# ---------------------------------------------------------------------

TRAIN_FRACTION = 0.70
VAL_FRACTION   = 0.15
TEST_FRACTION  = 0.15


# =====================================================================
# 3. TRAINING SETTINGS
# =====================================================================

SEED = 42

MAX_EPOCHS = 150

BATCH_SIZE = 64

LEARNING_RATE = 3e-3

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.03

DROPOUT = 0.30

BLOCK_DROPOUT = 0.05

GRAD_CLIP_NORM = 1.0


# ---------------------------------------------------------------------
# Early stopping
# ---------------------------------------------------------------------

EARLY_STOPPING_PATIENCE = 18

EARLY_STOPPING_MIN_DELTA = 1e-4


# ---------------------------------------------------------------------
# LR reduction
# ---------------------------------------------------------------------

LR_REDUCTION_FACTOR = 0.5

LR_PATIENCE = 5

MIN_LR = 1e-6


# =====================================================================
# 4. DATA AUGMENTATION
# =====================================================================
#
# Because the training data are synthetic, mild augmentation is useful
# to prevent the network from learning an unrealistically perfect
# McStas signature.
#
# We deliberately keep this mild.
#
# ---------------------------------------------------------------------

USE_AUGMENTATION = True

MAX_GAIN_CHANGE = 0.05

MAX_NOISE_STD = 0.010

MAX_BASELINE_SLOPE = 0.008


# =====================================================================
# 5. ROBUST INPUT PREPROCESSING
# =====================================================================
#
# Each diffraction pattern:
#
#   1. negative values -> 0
#   2. normalize to its 99.5th percentile
#   3. clip very extreme outliers
#   4. log compression
#
# This makes classification less dependent on absolute neutron flux
# and more dependent on peak structure / relative intensities.
#
# ---------------------------------------------------------------------

INTENSITY_PERCENTILE = 99.5

MAX_NORMALIZED_INTENSITY = 5.0

LOG_COMPRESSION = 20.0


# =====================================================================
# 6. DEVICE
# =====================================================================

if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif torch.backends.mps.is_available():

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )


# ---------------------------------------------------------------------
# CUDA AMP
#
# We enable mixed precision only on CUDA.
#
# On your Apple Silicon Mac, MPS will use normal float32 here.
# ---------------------------------------------------------------------

USE_AMP = (
    device.type == "cuda"
)


# ---------------------------------------------------------------------
# DataLoader workers
#
# num_workers=0 is deliberately safest inside Jupyter on macOS.
#
# For this small 1D dataset the loading overhead is negligible anyway.
# ---------------------------------------------------------------------

NUM_WORKERS = 0

PIN_MEMORY = (
    device.type == "cuda"
)


# =====================================================================
# 7. REPRODUCIBILITY
# =====================================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

    torch.backends.cudnn.benchmark = True



# =====================================================================
# 8. FIND DATASET
# =====================================================================

if DATASET_SWEEP is not None:

    sweep_root = Path(
        DATASET_SWEEP
    )

    # -------------------------------------------------------------
    # If only a folder name was given, interpret it relative to
    # DATASET_ROOT.
    # -------------------------------------------------------------

    if not sweep_root.is_absolute():

        sweep_root = (
            DATASET_ROOT
            /
            sweep_root
        )


    metadata_file = (
        sweep_root
        /
        "master_metadata.csv"
    )


    if not metadata_file.exists():

        raise FileNotFoundError(

            "\nCould not find:\n"
            f"{metadata_file.resolve()}"
        )


else:

    # -------------------------------------------------------------
    # DATASET_ROOT itself must exist
    # -------------------------------------------------------------

    if not DATASET_ROOT.exists():

        raise FileNotFoundError(

            "\nDataset directory does not exist:\n"
            f"{DATASET_ROOT.resolve()}"
        )


    # -------------------------------------------------------------
    # Look only one level below DATASET_ROOT for valid sweep folders
    #
    # Expected structure:
    #
    # DATASET_ROOT/
    #     sweep_1/
    #         master_metadata.csv
    #
    #     sweep_2/
    #         master_metadata.csv
    #
    # -------------------------------------------------------------

    metadata_candidates = sorted(

        DATASET_ROOT.glob(
            "*/master_metadata.csv"
        )
    )


    # -------------------------------------------------------------
    # No dataset found
    # -------------------------------------------------------------

    if len(metadata_candidates) == 0:

        raise FileNotFoundError(

            "\nNo valid dataset sweeps were found under:\n"
            f"{DATASET_ROOT.resolve()}"
        )


    # -------------------------------------------------------------
    # More than one dataset found
    #
    # Do NOT silently guess which one should be used.
    # -------------------------------------------------------------

    if len(metadata_candidates) > 1:

        print()
        print("Multiple dataset sweeps were found:")
        print()

        for i, candidate in enumerate(
            metadata_candidates,
            start=1
        ):

            print(
                f"{i}: {candidate.parent.name}"
            )


        raise RuntimeError(

            "\nMore than one dataset sweep exists.\n\n"
            "Set DATASET_SWEEP explicitly near the top of the notebook.\n\n"
            "For example:\n\n"
            "DATASET_SWEEP = Path(\n"
            '    "lambda_...__cfg_..."\n'
            ")\n"
        )


    # -------------------------------------------------------------
    # Exactly one valid sweep exists
    # -------------------------------------------------------------

    metadata_file = (
        metadata_candidates[0]
    )

    sweep_root = (
        metadata_file.parent
    )


# =====================================================================
# REPORT SELECTED DATASET
# =====================================================================

print("=" * 80)

print("DATASET")

print("=" * 80)

print()

print(
    "Dataset:"
)

print(
    sweep_root.resolve()
)

print()

print(
    "Metadata:"
)

print(
    metadata_file.resolve()
)

print()

# =====================================================================
# 9. OUTPUT DIRECTORY
# =====================================================================

output_dir = (

    sweep_root
    /
    "ML_material_classifier"
)


output_dir.mkdir(

    parents=True,
    exist_ok=True
)


checkpoint_file = (

    output_dir
    /
    "best_material_classifier.pt"
)


# =====================================================================
# 10. LOAD METADATA
# =====================================================================

df = pd.read_csv(
    metadata_file
)


required_columns = {

    "material",

    "lambda_A",

    "dlambda_A",

    "profile_1d_npz"
}


missing_columns = (

    required_columns
    -
    set(
        df.columns
    )
)


if missing_columns:

    raise ValueError(

        "\nMetadata is missing required columns:\n\n"

        + "\n".join(
            sorted(
                missing_columns
            )
        )
    )


# =====================================================================
# 11. CHECK PROFILE FILES
# =====================================================================

valid_rows = []


for idx, row in df.iterrows():

    profile_path = (

        sweep_root
        /
        row["profile_1d_npz"]
    )


    if profile_path.exists():

        valid_rows.append(
            idx
        )


df = (

    df
    .loc[
        valid_rows
    ]
    .reset_index(
        drop=True
    )
)


if len(df) == 0:

    raise RuntimeError(
        "\nNo valid 1D diffraction profiles were found."
    )


print()

print(
    f"Valid diffraction profiles: {len(df)}"
)


# =====================================================================
# 12. CLASS LABELS
# =====================================================================

class_names = list(

    pd.unique(
        df["material"]
    )
)


num_classes = len(
    class_names
)


class_to_idx = {

    name: i

    for i, name

    in enumerate(
        class_names
    )
}


idx_to_class = {

    i: name

    for name, i

    in class_to_idx.items()
}


df["class_index"] = (

    df["material"]
    .map(
        class_to_idx
    )
    .astype(
        int
    )
)


print()

print("=" * 80)

print(
    f"MATERIAL CLASSES: {num_classes}"
)

print("=" * 80)


for i, material in enumerate(
    class_names
):

    count = int(

        (
            df["material"]
            ==
            material
        ).sum()
    )

    print(

        f"{i:2d}  "
        f"{material:<20} "
        f"{count:5d} patterns"
    )


# =====================================================================
# 13. LOAD ALL 1D PROFILES
# =====================================================================

X_list = []

theta_reference = None


for row_number, row in df.iterrows():

    profile_path = (

        sweep_root
        /
        row["profile_1d_npz"]
    )


    profile = np.load(
        profile_path
    )


    if PROFILE_KEY not in profile.files:

        raise KeyError(

            f"\n'{PROFILE_KEY}' not found in:\n"

            f"{profile_path}\n\n"

            f"Available arrays:\n"

            f"{profile.files}"
        )


    signal = np.asarray(

        profile[
            PROFILE_KEY
        ],

        dtype=np.float64
    )


    theta = np.asarray(

        profile[
            "two_theta_deg"
        ],

        dtype=np.float64
    )


    if theta_reference is None:

        theta_reference = theta.copy()


    else:

        if (

            len(theta)
            !=
            len(theta_reference)

            or

            not np.allclose(

                theta,

                theta_reference,

                rtol=1e-6,

                atol=1e-6
            )
        ):

            raise ValueError(

                "\nNot all diffraction profiles use the same "
                "2theta coordinate system.\n\n"

                "A fixed-length CNN requires profiles on the "
                "same angular grid."
            )


    X_list.append(
        signal
    )


X_raw = np.stack(
    X_list,
    axis=0
)


y = df[
    "class_index"
].to_numpy(
    dtype=np.int64
)


print()

print(
    f"Raw X shape: {X_raw.shape}"
)

print(
    f"y shape:     {y.shape}"
)

print(
    f"2theta:      "
    f"{theta_reference.min():.3f}° "
    f"to "
    f"{theta_reference.max():.3f}°"
)


# =====================================================================
# 14. CHECK NaN / INF / EMPTY PROFILES
# =====================================================================

if not np.all(
    np.isfinite(
        X_raw
    )
):

    warnings.warn(

        "NaN or Inf values detected. "
        "They will be replaced by zero."
    )


X_raw = np.nan_to_num(

    X_raw,

    nan=0.0,

    posinf=0.0,

    neginf=0.0
)


empty_profiles = (

    np.max(
        np.abs(
            X_raw
        ),
        axis=1
    )
    <=
    0
)


if empty_profiles.any():

    print()

    print(
        f"WARNING: dropping "
        f"{empty_profiles.sum()} empty profiles."
    )


    keep = (
        ~empty_profiles
    )


    X_raw = X_raw[
        keep
    ]

    y = y[
        keep
    ]

    df = (

        df.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )


# =====================================================================
# 15. PREPROCESS EACH PROFILE
# =====================================================================

def preprocess_profile(
    signal
):

    x = np.asarray(

        signal,

        dtype=np.float64
    ).copy()


    x = np.nan_to_num(

        x,

        nan=0.0,

        posinf=0.0,

        neginf=0.0
    )


    # Physical detector intensity should be non-negative
    x = np.clip(

        x,

        0,

        None
    )


    positive = x[
        x > 0
    ]


    if positive.size == 0:

        return np.zeros_like(

            x,

            dtype=np.float32
        )


    scale = np.percentile(

        positive,

        INTENSITY_PERCENTILE
    )


    if (

        not np.isfinite(
            scale
        )

        or

        scale <= 0

    ):

        scale = positive.max()


    # -------------------------------------------------------------
    # Per-profile intensity normalization
    # -------------------------------------------------------------

    x = (

        x
        /
        scale
    )


    # -------------------------------------------------------------
    # Prevent isolated MC outliers from dominating
    # -------------------------------------------------------------

    x = np.clip(

        x,

        0,

        MAX_NORMALIZED_INTENSITY
    )


    # -------------------------------------------------------------
    # Log compression
    #
    # weak diffraction peaks become easier for the CNN to exploit
    # without destroying strong peaks.
    # -------------------------------------------------------------

    x = (

        np.log1p(

            LOG_COMPRESSION
            *
            x

        )

        /

        np.log1p(
            LOG_COMPRESSION
        )
    )


    return x.astype(
        np.float32
    )


X = np.stack(

    [

        preprocess_profile(
            row
        )

        for row in X_raw

    ],

    axis=0
)


# =====================================================================
# 16. CREATE CONDITION GROUPS
# =====================================================================
#
# THIS IS IMPORTANT.
#
# We group by:
#
#       lambda + dlambda
#
# not by individual pattern.
#
# =====================================================================

df["condition_group"] = [

    f"{lam:.10g}|{dlam:.10g}"

    for lam, dlam

    in zip(

        df["lambda_A"],

        df["dlambda_A"]
    )
]


# =====================================================================
# 17. REMOVE INCOMPLETE PARAMETER CONDITIONS
# =====================================================================
#
# Ideally every lambda/dlambda combination should contain every
# material.
#
# If an interrupted McStas sweep left incomplete combinations,
# remove those conditions rather than creating an imbalanced split.
#
# =====================================================================

group_class_counts = (

    df.groupby(
        "condition_group"
    )[
        "material"
    ]
    .nunique()
)


complete_groups = set(

    group_class_counts[

        group_class_counts
        ==
        num_classes

    ].index
)


incomplete_count = (

    len(
        group_class_counts
    )
    -
    len(
        complete_groups
    )
)


if incomplete_count > 0:

    print()

    print(
        f"Removing {incomplete_count} incomplete "
        f"(lambda, dlambda) conditions."
    )


    keep = (

        df[
            "condition_group"
        ]
        .isin(
            complete_groups
        )
        .to_numpy()
    )


    df = (

        df.loc[
            keep
        ]
        .reset_index(
            drop=True
        )
    )


    X = X[
        keep
    ]

    y = y[
        keep
    ]


# =====================================================================
# 18. GROUPED TRAIN / VALIDATION / TEST SPLIT
# =====================================================================

unique_groups = np.array(

    sorted(

        df[
            "condition_group"
        ]
        .unique()
    )
)


if len(unique_groups) < 6:

    raise RuntimeError(

        "\nToo few unique lambda/dlambda conditions "
        "for a robust train/validation/test split."
    )


rng = np.random.default_rng(
    SEED
)


rng.shuffle(
    unique_groups
)


n_groups = len(
    unique_groups
)


n_test = max(

    1,

    int(
        round(
            TEST_FRACTION
            *
            n_groups
        )
    )
)


n_val = max(

    1,

    int(
        round(
            VAL_FRACTION
            *
            n_groups
        )
    )
)


n_train = (

    n_groups
    -
    n_val
    -
    n_test
)


if n_train < 1:

    raise RuntimeError(
        "Not enough groups for training."
    )


train_groups = set(

    unique_groups[
        :n_train
    ]
)


val_groups = set(

    unique_groups[
        n_train:
        n_train + n_val
    ]
)


test_groups = set(

    unique_groups[
        n_train + n_val:
    ]
)


train_idx = np.where(

    df[
        "condition_group"
    ]
    .isin(
        train_groups
    )
    .to_numpy()

)[0]


val_idx = np.where(

    df[
        "condition_group"
    ]
    .isin(
        val_groups
    )
    .to_numpy()

)[0]


test_idx = np.where(

    df[
        "condition_group"
    ]
    .isin(
        test_groups
    )
    .to_numpy()

)[0]


# =====================================================================
# 19. VERIFY ABSOLUTELY NO CONDITION LEAKAGE
# =====================================================================

assert train_groups.isdisjoint(
    val_groups
)

assert train_groups.isdisjoint(
    test_groups
)

assert val_groups.isdisjoint(
    test_groups
)


# =====================================================================
# 20. VERIFY ALL CLASSES OCCUR IN ALL SPLITS
# =====================================================================

def verify_classes(
    indices,
    name
):

    present = set(

        y[
            indices
        ].tolist()
    )


    expected = set(

        range(
            num_classes
        )
    )


    if present != expected:

        missing = expected - present

        raise RuntimeError(

            f"\n{name} split is missing classes:\n"

            f"{[idx_to_class[i] for i in missing]}"
        )


verify_classes(
    train_idx,
    "TRAIN"
)

verify_classes(
    val_idx,
    "VALIDATION"
)

verify_classes(
    test_idx,
    "TEST"
)


# =====================================================================
# 21. TRAINING-ONLY STANDARDIZATION
# =====================================================================
#
# We deliberately calculate normalization statistics from TRAINING ONLY.
#
# No information from validation or test enters preprocessing.
#
# =====================================================================

train_global_mean = float(

    X[
        train_idx
    ].mean()
)


train_global_std = float(

    X[
        train_idx
    ].std()
)


train_global_std = max(

    train_global_std,

    1e-6
)


print()

print("=" * 80)
print("DATA SPLIT")
print("=" * 80)

print()

print(
    f"Parameter conditions:"
)

print(
    f"    train      = {len(train_groups)}"
)

print(
    f"    validation = {len(val_groups)}"
)

print(
    f"    test       = {len(test_groups)}"
)

print()

print(
    f"Profiles:"
)

print(
    f"    train      = {len(train_idx)}"
)

print(
    f"    validation = {len(val_idx)}"
)

print(
    f"    test       = {len(test_idx)}"
)

print()

print(
    f"Training mean = {train_global_mean:.6f}"
)

print(
    f"Training std  = {train_global_std:.6f}"
)


# =====================================================================
# 22. SAVE SPLIT ASSIGNMENTS
# =====================================================================

df["split"] = "unused"

df.loc[
    train_idx,
    "split"
] = "train"

df.loc[
    val_idx,
    "split"
] = "validation"

df.loc[
    test_idx,
    "split"
] = "test"


df.to_csv(

    output_dir
    /
    "dataset_split.csv",

    index=False
)


# =====================================================================
# 23. DATASET CLASS
# =====================================================================

class DiffractionDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y,
        indices,
        mean,
        std,
        augment=False
    ):

        self.X = X

        self.y = y

        self.indices = np.asarray(
            indices
        )

        self.mean = float(
            mean
        )

        self.std = float(
            std
        )

        self.augment = augment


    def __len__(
        self
    ):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        dataset_index
    ):

        original_index = int(

            self.indices[
                dataset_index
            ]
        )


        x = torch.tensor(

            self.X[
                original_index
            ],

            dtype=torch.float32
        )


        # =============================================================
        # TRAINING AUGMENTATION
        # =============================================================

        if self.augment:


            # ---------------------------------------------------------
            # Small overall intensity gain variation
            # ---------------------------------------------------------

            if torch.rand(
                1
            ).item() < 0.8:

                gain = (

                    1.0
                    +
                    (
                        2.0
                        *
                        torch.rand(
                            1
                        ).item()
                        -
                        1.0
                    )
                    *
                    MAX_GAIN_CHANGE
                )


                x = (
                    x
                    *
                    gain
                )


            # ---------------------------------------------------------
            # Mild detector/statistical noise
            # ---------------------------------------------------------

            if torch.rand(
                1
            ).item() < 0.8:

                noise_std = (

                    torch.rand(
                        1
                    ).item()
                    *
                    MAX_NOISE_STD
                )


                x = (

                    x
                    +
                    torch.randn_like(
                        x
                    )
                    *
                    noise_std
                )


            # ---------------------------------------------------------
            # Very mild smooth baseline slope
            # ---------------------------------------------------------

            if torch.rand(
                1
            ).item() < 0.35:

                slope = (

                    (
                        2.0
                        *
                        torch.rand(
                            1
                        ).item()
                        -
                        1.0
                    )
                    *
                    MAX_BASELINE_SLOPE
                )


                baseline_axis = torch.linspace(

                    -1,

                    1,

                    len(
                        x
                    )
                )


                x = (

                    x
                    +
                    slope
                    *
                    baseline_axis
                )


            x = torch.clamp(

                x,

                min=0
            )


        # =============================================================
        # TRAINING-ONLY STANDARDIZATION
        # =============================================================

        x = (

            x
            -
            self.mean

        ) / self.std


        # CNN input:
        #
        #       [channels, signal_length]
        #
        # so:
        #
        #       [1, 600]
        #
        x = x.unsqueeze(
            0
        )


        target = torch.tensor(

            self.y[
                original_index
            ],

            dtype=torch.long
        )


        return (

            x,

            target,

            original_index
        )


# =====================================================================
# 24. DATASETS
# =====================================================================

train_dataset = DiffractionDataset(

    X,
    y,
    train_idx,

    train_global_mean,
    train_global_std,

    augment=USE_AUGMENTATION
)


val_dataset = DiffractionDataset(

    X,
    y,
    val_idx,

    train_global_mean,
    train_global_std,

    augment=False
)


test_dataset = DiffractionDataset(

    X,
    y,
    test_idx,

    train_global_mean,
    train_global_std,

    augment=False
)


# =====================================================================
# 25. DATALOADERS
# =====================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY
)


# =====================================================================
# 26. SQUEEZE-AND-EXCITATION BLOCK
# =====================================================================

class SEBlock1D(
    nn.Module
):

    def __init__(
        self,
        channels,
        reduction=8
    ):

        super().__init__()


        hidden = max(

            channels
            //
            reduction,

            8
        )


        self.pool = nn.AdaptiveAvgPool1d(
            1
        )


        self.fc = nn.Sequential(

            nn.Conv1d(
                channels,
                hidden,
                kernel_size=1
            ),

            nn.SiLU(),

            nn.Conv1d(
                hidden,
                channels,
                kernel_size=1
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        x
    ):

        weights = self.fc(

            self.pool(
                x
            )
        )


        return (
            x
            *
            weights
        )


# =====================================================================
# 27. RESIDUAL BLOCK
# =====================================================================

class ResidualBlock1D(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        dropout=0.0
    ):

        super().__init__()


        padding = (
            kernel_size
            //
            2
        )


        self.conv1 = nn.Conv1d(

            in_channels,

            out_channels,

            kernel_size=kernel_size,

            stride=stride,

            padding=padding,

            bias=False
        )


        self.bn1 = nn.BatchNorm1d(
            out_channels
        )


        self.act = nn.SiLU()


        self.conv2 = nn.Conv1d(

            out_channels,

            out_channels,

            kernel_size=kernel_size,

            padding=padding,

            bias=False
        )


        self.bn2 = nn.BatchNorm1d(
            out_channels
        )


        self.dropout = nn.Dropout1d(
            dropout
        )


        self.se = SEBlock1D(
            out_channels
        )


        if (

            stride != 1

            or

            in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(

                nn.Conv1d(

                    in_channels,

                    out_channels,

                    kernel_size=1,

                    stride=stride,

                    bias=False
                ),

                nn.BatchNorm1d(
                    out_channels
                )
            )


        else:

            self.shortcut = nn.Identity()


    def forward(
        self,
        x
    ):

        identity = self.shortcut(
            x
        )


        out = self.conv1(
            x
        )

        out = self.bn1(
            out
        )

        out = self.act(
            out
        )


        out = self.dropout(
            out
        )


        out = self.conv2(
            out
        )

        out = self.bn2(
            out
        )


        out = self.se(
            out
        )


        out = (

            out
            +
            identity
        )


        out = self.act(
            out
        )


        return out


# =====================================================================
# 28. 1D RESIDUAL MATERIAL CLASSIFIER
# =====================================================================

class DiffractionResNet1D(
    nn.Module
):

    def __init__(
        self,
        num_classes,
        dropout=0.30
    ):

        super().__init__()


        # -------------------------------------------------------------
        # Initial broad receptive field
        # -------------------------------------------------------------

        self.stem = nn.Sequential(

            nn.Conv1d(

                1,

                32,

                kernel_size=15,

                stride=2,

                padding=7,

                bias=False
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.SiLU()
        )


        # -------------------------------------------------------------
        # Multi-scale hierarchy
        #
        # broad kernels first -> narrower kernels deeper
        # -------------------------------------------------------------

        self.features = nn.Sequential(


            ResidualBlock1D(

                32,
                32,

                kernel_size=7,

                stride=1,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                32,
                64,

                kernel_size=7,

                stride=2,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                64,
                64,

                kernel_size=5,

                stride=1,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                64,
                128,

                kernel_size=5,

                stride=2,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                128,
                128,

                kernel_size=3,

                stride=1,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                128,
                192,

                kernel_size=3,

                stride=2,

                dropout=BLOCK_DROPOUT
            ),


            ResidualBlock1D(

                192,
                192,

                kernel_size=3,

                stride=1,

                dropout=BLOCK_DROPOUT
            )
        )


        # -------------------------------------------------------------
        # Use BOTH global average and max pooling.
        #
        # Average:
        #       distributed pattern information
        #
        # Max:
        #       strong localized diffraction peaks
        # -------------------------------------------------------------

        self.avg_pool = nn.AdaptiveAvgPool1d(
            1
        )

        self.max_pool = nn.AdaptiveMaxPool1d(
            1
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                192 * 2,
                128
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        x = self.stem(
            x
        )

        x = self.features(
            x
        )


        avg = self.avg_pool(
            x
        )

        maximum = self.max_pool(
            x
        )


        x = torch.cat(

            [
                avg,
                maximum
            ],

            dim=1
        )


        logits = self.classifier(
            x
        )


        return logits


# =====================================================================
# 29. CREATE MODEL
# =====================================================================

model = DiffractionResNet1D(

    num_classes=num_classes,

    dropout=DROPOUT
)


model = model.to(
    device
)


number_of_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print()

print("=" * 80)
print("MODEL")
print("=" * 80)

print()

print(
    f"Device:                {device}"
)

print(
    f"Input length:          {X.shape[1]}"
)

print(
    f"Classes:               {num_classes}"
)

print(
    f"Trainable parameters:  {number_of_parameters:,}"
)

print(
    f"CUDA mixed precision:  {USE_AMP}"
)


# =====================================================================
# 30. CLASS BALANCING
# =====================================================================

train_counts = np.bincount(

    y[
        train_idx
    ],

    minlength=num_classes
)


imbalance_ratio = (

    train_counts.max()
    /
    max(
        train_counts.min(),
        1
    )
)


if imbalance_ratio > 1.20:

    class_weights = (

        train_counts.sum()

        /

        (
            num_classes
            *
            train_counts
        )
    )


    class_weights = torch.tensor(

        class_weights,

        dtype=torch.float32,

        device=device
    )


    print()

    print(
        "Class imbalance detected -> "
        "using weighted loss."
    )


else:

    class_weights = None


# =====================================================================
# 31. LOSS
# =====================================================================

criterion = nn.CrossEntropyLoss(

    weight=class_weights,

    label_smoothing=LABEL_SMOOTHING
)


# =====================================================================
# 32. OPTIMIZER
# =====================================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# =====================================================================
# 33. LEARNING RATE SCHEDULER
# =====================================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=LR_REDUCTION_FACTOR,

    patience=LR_PATIENCE,

    min_lr=MIN_LR
)


# =====================================================================
# 34. CUDA GRAD SCALER
# =====================================================================

try:

    scaler = torch.amp.GradScaler(

        "cuda",

        enabled=USE_AMP
    )


except Exception:

    scaler = torch.cuda.amp.GradScaler(

        enabled=USE_AMP
    )


# =====================================================================
# 35. AUTOCAST CONTEXT
# =====================================================================

def amp_context():

    if USE_AMP:

        return torch.autocast(

            device_type="cuda",

            dtype=torch.float16
        )


    return contextlib.nullcontext()


# =====================================================================
# 36. ONE EPOCH FUNCTION
# =====================================================================

def run_epoch(
    model,
    loader,
    training=False
):

    if training:

        model.train()

    else:

        model.eval()


    running_loss = 0.0

    all_targets = []

    all_predictions = []


    for inputs, targets, _ in loader:


        inputs = inputs.to(

            device,

            non_blocking=True
        )


        targets = targets.to(

            device,

            non_blocking=True
        )


        if training:

            optimizer.zero_grad(

                set_to_none=True
            )


        with torch.set_grad_enabled(
            training
        ):


            with amp_context():


                logits = model(
                    inputs
                )


                loss = criterion(

                    logits,

                    targets
                )


            if training:


                if USE_AMP:


                    scaler.scale(
                        loss
                    ).backward()


                    scaler.unscale_(
                        optimizer
                    )


                    torch.nn.utils.clip_grad_norm_(

                        model.parameters(),

                        GRAD_CLIP_NORM
                    )


                    scaler.step(
                        optimizer
                    )


                    scaler.update()


                else:


                    loss.backward()


                    torch.nn.utils.clip_grad_norm_(

                        model.parameters(),

                        GRAD_CLIP_NORM
                    )


                    optimizer.step()


        batch_size = inputs.size(
            0
        )


        running_loss += (

            loss.item()
            *
            batch_size
        )


        predictions = torch.argmax(

            logits,

            dim=1
        )


        all_targets.extend(

            targets
            .detach()
            .cpu()
            .numpy()
        )


        all_predictions.extend(

            predictions
            .detach()
            .cpu()
            .numpy()
        )


    all_targets = np.asarray(
        all_targets
    )


    all_predictions = np.asarray(
        all_predictions
    )


    epoch_loss = (

        running_loss
        /
        len(
            loader.dataset
        )
    )


    epoch_accuracy = accuracy_score(

        all_targets,

        all_predictions
    )


    epoch_f1 = f1_score(

        all_targets,

        all_predictions,

        average="macro",

        zero_division=0
    )


    return (

        epoch_loss,

        epoch_accuracy,

        epoch_f1
    )


# =====================================================================
# 37. TRAINING LOOP
# =====================================================================

history = []


best_val_f1 = -np.inf

best_val_loss = np.inf


epochs_without_improvement = 0


training_start = time.time()


print()

print("=" * 80)
print("TRAINING")
print("=" * 80)

print()


for epoch in range(

    1,

    MAX_EPOCHS + 1

):


    # -------------------------------------------------------------
    # TRAIN
    # -------------------------------------------------------------

    (

        train_loss,

        train_accuracy,

        train_f1

    ) = run_epoch(

        model,

        train_loader,

        training=True
    )


    # -------------------------------------------------------------
    # VALIDATE
    # -------------------------------------------------------------

    (

        val_loss,

        val_accuracy,

        val_f1

    ) = run_epoch(

        model,

        val_loader,

        training=False
    )


    # -------------------------------------------------------------
    # LR scheduler
    # -------------------------------------------------------------

    scheduler.step(
        val_loss
    )


    current_lr = optimizer.param_groups[
        0
    ][
        "lr"
    ]


    # -------------------------------------------------------------
    # SAVE HISTORY
    # -------------------------------------------------------------

    history.append(

        {

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "train_macro_f1":
                train_f1,

            "val_macro_f1":
                val_f1,

            "learning_rate":
                current_lr
        }
    )


    # -------------------------------------------------------------
    # Did validation improve?
    #
    # Main criterion:
    #
    #       macro F1
    #
    # tie breaker:
    #
    #       lower validation loss
    # -------------------------------------------------------------

    improved = (

        val_f1
        >
        (
            best_val_f1
            +
            EARLY_STOPPING_MIN_DELTA
        )
    )


    tied_f1_better_loss = (

        abs(
            val_f1
            -
            best_val_f1
        )
        <=
        EARLY_STOPPING_MIN_DELTA

        and

        val_loss
        <
        (
            best_val_loss
            -
            1e-4
        )
    )


    if (

        improved

        or

        tied_f1_better_loss

    ):


        best_val_f1 = val_f1

        best_val_loss = val_loss

        epochs_without_improvement = 0


        checkpoint = {

            "model_state_dict":
                model.state_dict(),

            "class_names":
                class_names,

            "class_to_idx":
                class_to_idx,

            "theta_deg":
                theta_reference.astype(
                    np.float32
                ),

            "train_global_mean":
                train_global_mean,

            "train_global_std":
                train_global_std,

            "profile_key":
                PROFILE_KEY,

            "intensity_percentile":
                INTENSITY_PERCENTILE,

            "max_normalized_intensity":
                MAX_NORMALIZED_INTENSITY,

            "log_compression":
                LOG_COMPRESSION,

            "num_classes":
                num_classes,

            "input_length":
                X.shape[1],

            "seed":
                SEED,

            "epoch":
                epoch,

            "val_macro_f1":
                val_f1,

            "val_loss":
                val_loss
        }


        torch.save(

            checkpoint,

            checkpoint_file
        )


        marker = "  ← BEST"


    else:

        epochs_without_improvement += 1

        marker = ""


    print(

        f"Epoch "
        f"{epoch:03d}/{MAX_EPOCHS} | "

        f"loss "
        f"{train_loss:.4f}/"
        f"{val_loss:.4f} | "

        f"acc "
        f"{train_accuracy:.4f}/"
        f"{val_accuracy:.4f} | "

        f"F1 "
        f"{train_f1:.4f}/"
        f"{val_f1:.4f} | "

        f"LR "
        f"{current_lr:.2e}"

        f"{marker}"
    )


    # -------------------------------------------------------------
    # EARLY STOPPING
    # -------------------------------------------------------------

    if (

        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE

    ):

        print()

        print(
            f"Early stopping at epoch {epoch}."
        )

        print(
            f"Best validation macro-F1 = "
            f"{best_val_f1:.4f}"
        )

        break


training_seconds = (

    time.time()
    -
    training_start
)


# =====================================================================
# 38. SAVE TRAINING HISTORY
# =====================================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    output_dir
    /
    "training_history.csv",

    index=False
)


# =====================================================================
# 39. LOAD BEST MODEL
# =====================================================================

try:

    checkpoint = torch.load(

        checkpoint_file,

        map_location=device,

        weights_only=False
    )


except TypeError:

    checkpoint = torch.load(

        checkpoint_file,

        map_location=device
    )


model.load_state_dict(

    checkpoint[
        "model_state_dict"
    ]
)


model.eval()


print()

print("=" * 80)

print("BEST MODEL")

print("=" * 80)

print()

print(
    f"Best epoch:       "
    f"{checkpoint['epoch']}"
)

print(
    f"Validation F1:    "
    f"{checkpoint['val_macro_f1']:.4f}"
)

print(
    f"Validation loss:  "
    f"{checkpoint['val_loss']:.4f}"
)

print(
    f"Training time:    "
    f"{training_seconds:.1f} s"
)


# =====================================================================
# 40. FULL TEST EVALUATION
# =====================================================================

all_test_targets = []

all_test_predictions = []

all_test_probabilities = []

all_test_indices = []


model.eval()


with torch.no_grad():


    for inputs, targets, original_indices in test_loader:


        inputs = inputs.to(

            device,

            non_blocking=True
        )


        with amp_context():


            logits = model(
                inputs
            )


        probabilities = torch.softmax(

            logits,

            dim=1
        )


        predictions = torch.argmax(

            probabilities,

            dim=1
        )


        all_test_targets.extend(

            targets.numpy()
        )


        all_test_predictions.extend(

            predictions
            .cpu()
            .numpy()
        )


        all_test_probabilities.append(

            probabilities
            .cpu()
            .numpy()
        )


        all_test_indices.extend(

            original_indices.numpy()
        )


y_true = np.asarray(

    all_test_targets,

    dtype=int
)


y_pred = np.asarray(

    all_test_predictions,

    dtype=int
)


y_prob = np.concatenate(

    all_test_probabilities,

    axis=0
)


test_original_indices = np.asarray(

    all_test_indices,

    dtype=int
)


# =====================================================================
# 41. TEST METRICS
# =====================================================================

test_accuracy = accuracy_score(

    y_true,

    y_pred
)


test_balanced_accuracy = balanced_accuracy_score(

    y_true,

    y_pred
)


test_macro_f1 = f1_score(

    y_true,

    y_pred,

    average="macro",

    zero_division=0
)


test_weighted_f1 = f1_score(

    y_true,

    y_pred,

    average="weighted",

    zero_division=0
)


# ---------------------------------------------------------------------
# Top-2 accuracy
# ---------------------------------------------------------------------

if num_classes >= 2:

    top2 = np.argsort(

        y_prob,

        axis=1
    )[
        :,
        -2:
    ]


    top2_accuracy = np.mean(

        [

            true_class
            in
            top_classes

            for true_class, top_classes

            in zip(

                y_true,

                top2
            )
        ]
    )


else:

    top2_accuracy = 1.0


# =====================================================================
# 42. EXPECTED CALIBRATION ERROR
# =====================================================================
#
# Accuracy alone does not tell us whether confidence is trustworthy.
#
# ECE compares predicted confidence with actual correctness.
#
# Lower is better.
#
# =====================================================================

def expected_calibration_error(

    probabilities,

    targets,

    n_bins=10
):

    confidences = probabilities.max(
        axis=1
    )


    predictions = probabilities.argmax(
        axis=1
    )


    correct = (

        predictions
        ==
        targets
    ).astype(
        float
    )


    bin_edges = np.linspace(

        0,

        1,

        n_bins + 1
    )


    ece = 0.0


    for i in range(
        n_bins
    ):

        lower = bin_edges[
            i
        ]

        upper = bin_edges[
            i + 1
        ]


        if i == n_bins - 1:

            mask = (

                (confidences >= lower)

                &

                (confidences <= upper)
            )

        else:

            mask = (

                (confidences >= lower)

                &

                (confidences < upper)
            )


        if mask.sum() == 0:

            continue


        bin_accuracy = correct[
            mask
        ].mean()


        bin_confidence = confidences[
            mask
        ].mean()


        ece += (

            mask.mean()

            *

            abs(

                bin_accuracy

                -

                bin_confidence
            )
        )


    return float(
        ece
    )


test_ece = expected_calibration_error(

    y_prob,

    y_true
)


# =====================================================================
# 43. CLASSIFICATION REPORT
# =====================================================================

report_dict = classification_report(

    y_true,

    y_pred,

    labels=np.arange(
        num_classes
    ),

    target_names=class_names,

    output_dict=True,

    zero_division=0
)


report_df = pd.DataFrame(

    report_dict
).T


report_df.to_csv(

    output_dir
    /
    "classification_report.csv"
)


# =====================================================================
# 44. CONFUSION MATRIX
# =====================================================================

cm = confusion_matrix(

    y_true,

    y_pred,

    labels=np.arange(
        num_classes
    )
)


cm_normalized = confusion_matrix(

    y_true,

    y_pred,

    labels=np.arange(
        num_classes
    ),

    normalize="true"
)


# =====================================================================
# 45. SAVE TEST PREDICTIONS
# =====================================================================

test_results = (

    df.iloc[
        test_original_indices
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


test_results[
    "true_material"
] = [

    idx_to_class[
        int(i)
    ]

    for i in y_true
]


test_results[
    "predicted_material"
] = [

    idx_to_class[
        int(i)
    ]

    for i in y_pred
]


test_results[
    "confidence"
] = y_prob.max(
    axis=1
)


# ---------------------------------------------------------------------
# second best prediction
# ---------------------------------------------------------------------

if num_classes >= 2:

    sorted_classes = np.argsort(

        y_prob,

        axis=1
    )


    second_idx = sorted_classes[
        :,
        -2
    ]


    test_results[
        "second_prediction"
    ] = [

        idx_to_class[
            int(i)
        ]

        for i in second_idx
    ]


    test_results[
        "second_confidence"
    ] = [

        y_prob[
            row,
            cls
        ]

        for row, cls

        in enumerate(
            second_idx
        )
    ]


test_results.to_csv(

    output_dir
    /
    "test_predictions.csv",

    index=False
)


# =====================================================================
# 46. SAVE METRICS
# =====================================================================

metrics = {

    "test_accuracy":
        float(
            test_accuracy
        ),

    "test_balanced_accuracy":
        float(
            test_balanced_accuracy
        ),

    "test_macro_f1":
        float(
            test_macro_f1
        ),

    "test_weighted_f1":
        float(
            test_weighted_f1
        ),

    "test_top2_accuracy":
        float(
            top2_accuracy
        ),

    "test_expected_calibration_error":
        float(
            test_ece
        ),

    "best_validation_macro_f1":
        float(
            checkpoint[
                "val_macro_f1"
            ]
        ),

    "best_epoch":
        int(
            checkpoint[
                "epoch"
            ]
        ),

    "number_of_classes":
        int(
            num_classes
        ),

    "train_samples":
        int(
            len(
                train_idx
            )
        ),

    "validation_samples":
        int(
            len(
                val_idx
            )
        ),

    "test_samples":
        int(
            len(
                test_idx
            )
        ),

    "train_conditions":
        int(
            len(
                train_groups
            )
        ),

    "validation_conditions":
        int(
            len(
                val_groups
            )
        ),

    "test_conditions":
        int(
            len(
                test_groups
            )
        )
}


with open(

    output_dir
    /
    "test_metrics.json",

    "w"

) as f:

    json.dump(

        metrics,

        f,

        indent=4
    )


# =====================================================================
# 47. PRINT TEST RESULTS
# =====================================================================

print()

print("=" * 80)

print("FINAL HELD-OUT TEST SET")

print("=" * 80)

print()

print(
    f"Accuracy:             "
    f"{test_accuracy:.4f}"
)

print(
    f"Balanced accuracy:    "
    f"{test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:             "
    f"{test_macro_f1:.4f}"
)

print(
    f"Weighted F1:          "
    f"{test_weighted_f1:.4f}"
)

print(
    f"Top-2 accuracy:       "
    f"{top2_accuracy:.4f}"
)

print(
    f"Calibration error:    "
    f"{test_ece:.4f}"
)

print()

print(
    "Per-material results:"
)

display(

    report_df.loc[
        class_names
    ]
)


# =====================================================================
# 48. TRAINING CURVES
# =====================================================================

fig = plt.figure(

    figsize=(
        9,
        5
    )
)


plt.plot(

    history_df[
        "epoch"
    ],

    history_df[
        "train_loss"
    ],

    label="Train"
)


plt.plot(

    history_df[
        "epoch"
    ],

    history_df[
        "val_loss"
    ],

    label="Validation"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Cross-entropy loss"
)

plt.title(
    "Training and validation loss"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()


plt.savefig(

    output_dir
    /
    "loss_curve.png",

    dpi=160
)


plt.show()


# ---------------------------------------------------------------------

fig = plt.figure(

    figsize=(
        9,
        5
    )
)


plt.plot(

    history_df[
        "epoch"
    ],

    history_df[
        "train_macro_f1"
    ],

    label="Train"
)


plt.plot(

    history_df[
        "epoch"
    ],

    history_df[
        "val_macro_f1"
    ],

    label="Validation"
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "Training and validation macro-F1"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.tight_layout()


plt.savefig(

    output_dir
    /
    "f1_curve.png",

    dpi=160
)


plt.show()


# =====================================================================
# 49. NORMALIZED CONFUSION MATRIX
# =====================================================================

fig = plt.figure(

    figsize=(
        10,
        8
    )
)


plt.imshow(
    cm_normalized
)


plt.colorbar(
    label="Fraction of true class"
)


plt.xticks(

    np.arange(
        num_classes
    ),

    class_names,

    rotation=45,

    ha="right"
)


plt.yticks(

    np.arange(
        num_classes
    ),

    class_names
)


plt.xlabel(
    "Predicted material"
)

plt.ylabel(
    "True material"
)

plt.title(
    "Normalized confusion matrix — held-out test set"
)


for i in range(
    num_classes
):

    for j in range(
        num_classes
    ):

        value = cm_normalized[
            i,
            j
        ]


        if value >= 0.01:

            plt.text(

                j,
                i,

                f"{value:.2f}",

                ha="center",

                va="center"
            )


plt.tight_layout()


plt.savefig(

    output_dir
    /
    "confusion_matrix_normalized.png",

    dpi=180
)


plt.show()


# =====================================================================
# 50. RAW CONFUSION MATRIX
# =====================================================================

fig = plt.figure(

    figsize=(
        10,
        8
    )
)


plt.imshow(
    cm
)


plt.colorbar(
    label="Number of patterns"
)


plt.xticks(

    np.arange(
        num_classes
    ),

    class_names,

    rotation=45,

    ha="right"
)


plt.yticks(

    np.arange(
        num_classes
    ),

    class_names
)


plt.xlabel(
    "Predicted material"
)

plt.ylabel(
    "True material"
)

plt.title(
    "Confusion matrix — held-out test set"
)


for i in range(
    num_classes
):

    for j in range(
        num_classes
    ):

        value = cm[
            i,
            j
        ]


        if value > 0:

            plt.text(

                j,
                i,

                str(
                    value
                ),

                ha="center",

                va="center"
            )


plt.tight_layout()


plt.savefig(

    output_dir
    /
    "confusion_matrix_counts.png",

    dpi=180
)


plt.show()


# =====================================================================
# 51. EXAMPLE TEST PREDICTIONS
# =====================================================================

print()

print("=" * 80)

print("EXAMPLE TEST PREDICTIONS")

print("=" * 80)

display(

    test_results[
        [

            "material",

            "lambda_A",

            "dlambda_A",

            "predicted_material",

            "confidence"

        ]
    ]
    .head(
        20
    )
)


# =====================================================================
# 52. SUMMARY OF EVERYTHING SAVED
# =====================================================================

print()

print("=" * 80)

print("TRAINING COMPLETE")

print("=" * 80)

print()

print(
    "Best model:"
)

print(
    checkpoint_file.resolve()
)

print()

print(
    "All ML outputs:"
)

print(
    output_dir.resolve()
)

print()

print(
    "Saved files include:"
)

print(
    "  best_material_classifier.pt"
)

print(
    "  dataset_split.csv"
)

print(
    "  training_history.csv"
)

print(
    "  classification_report.csv"
)

print(
    "  test_predictions.csv"
)

print(
    "  test_metrics.json"
)

print(
    "  loss_curve.png"
)

print(
    "  f1_curve.png"
)

print(
    "  confusion_matrix_normalized.png"
)

print(
    "  confusion_matrix_counts.png"
)

print()

print("=" * 80)